In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
import numpy as np
from scipy.optimize import differential_evolution

# Check if we have a GPU available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
""" Image Preprocessing - CIFAR-10 normalization values """

# CIFAR-10 channel-wise normalization (RGB)
mean_vals = [0.4914, 0.4822, 0.4465]
std_vals = [0.2023, 0.1994, 0.2010] 

mean = torch.tensor(mean_vals).view(3, 1, 1)
std = torch.tensor(std_vals).view(3, 1, 1)

# Simple transform: convert images to tensors without augmentation
transform_raw = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

# Load CIFAR-10 test dataset
testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_raw
)

# Create data loader with batch size 1 for single image processing
testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=True)

# Helper to build CIFAR-10 compatible ResNet-18
def make_cifar_resnet18():
    # Start with standard ResNet-18 architecture
    model = torchvision.models.resnet18(weights=None)
    
    # Modify first conv layer for smaller input (3x32x32)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    
    # Adjust max pooling to avoid over-downsampling on small images
    model.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    
    # Replace classification head for 10 CIFAR classes
    model.fc = nn.Linear(512, 10)

    # Load pretrained weights 
    state_dict = torch.load("resnet18.pt", map_location="cpu")
    model.load_state_dict(state_dict)
    
    return model

# Load model and prepare for inference
model = make_cifar_resnet18()
model.to(device).eval();

In [6]:
# CIFAR-10 class labels in order (0-9)
CIFAR10_CLASSES = [ "airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

@torch.no_grad() # No gradient (only used for training)
def predict(model, img_norm):
    """Returns prediction and confidence for a normalized image."""
    # Forward pass through model
    logits = model(img_norm.unsqueeze(0))
    
    # Convert logits to probabilities
    probs = torch.softmax(logits, dim=1)[0]
    
    # Get class with highest probability
    pred = probs.argmax().item()
    
    # Get confidence score for predicted class
    conf = probs[pred].item()
    return pred, conf

def get_fitness_function(img_norm, label, model, mean, std, n_pixels=1):
    """
    Creates and returns a fitness function for n-pixel adversarial attack.
    
    The optimizer works in [0, 255] RGB space; we convert to normalized space
    for model evaluation. Based on Su et al. (2017) one-pixel attack.
    
    Args:
        img_norm: Normalized image tensor [C, H, W]
        label: Ground truth class label
        model: Target model for attack
        mean, std: Normalization statistics for denorm/renorm conversion
        n_pixels: Number of pixels to perturb
    """
    def fitness_and_pred(candidate):
        # Create adversary variable - a copy of the original image
        adv = img_norm.clone()
        
        # Modify each target pixel
        for i in range(n_pixels):
            # Candidate vector: [x, y, r, g, b] for each pixel
            idx = i * 5
            x_pos = int(round(candidate[idx]))
            y_pos = int(round(candidate[idx + 1]))
            
            # Extract RGB values from optimizer (in [0, 255] range)
            r_255 = candidate[idx + 2]
            g_255 = candidate[idx + 3]
            b_255 = candidate[idx + 4]
            
            # Convert RGB from [0, 255] to normalized space using CIFAR-10 stats
            r_norm = (r_255 / 255.0 - mean[0]) / std[0]
            g_norm = (g_255 / 255.0 - mean[1]) / std[1]
            b_norm = (b_255 / 255.0 - mean[2]) / std[2]
            
            # Set pixel in adversarial image
            adv[0, y_pos, x_pos] = r_norm
            adv[1, y_pos, x_pos] = g_norm
            adv[2, y_pos, x_pos] = b_norm
        
        # Evaluate modified image
        with torch.no_grad():
            logits = model(adv.unsqueeze(0))
            probs = torch.softmax(logits, dim=1)[0]
        
        pred = probs.argmax().item()
        conf = probs[pred].item()
        fit = probs[label].item()  # How confident is model in true class
        
        return fit, pred, conf, adv
    
    return fitness_and_pred

In [10]:
def differential_evolution_attack_n_pixels(img_norm, label, model, mean, std,
                                          n_pixels=1, popsize=80, maxiter=60):
    """
    Runs n-pixel attack using differential evolutionl.
    
    Searches for minimal pixel perturbations that fool the model. RGB values
    are optimized in [0, 255] space following Su et al. (2017).
    
    Arguments:
        img_norm: Normalized image tensor [C, H, W]
        label: correct class label
        model: model to attack
        mean, std: CIFAR-10 normalization statistics
        n_pixels: Number of pixels to perturb (1, 2 or 3)
        popsize: Population size for DE optimizer
        maxiter: Maximum generations to run
    
    Returns:
        pixels: List of (x, y, r, g, b) tuples for modified pixels
        iterations: Actual number of generations used
        final_fitness: Probability of correct class after attack
        final_pred: Model's prediction on adversarial image
        final_conf: Confidence in adversarial prediction
    """
    # Define search bounds for each pixel: (x, y, R, G, B)
    pixel_bounds = [
        (0, 31),   # x coordinate (CIFAR-10 is 32x32)
        (0, 31),   # y coordinate
        (0, 255),  # Red channel value
        (0, 255),  # Green channel value
        (0, 255),  # Blue channel value
    ]
    
    bounds = pixel_bounds * n_pixels
    
    fitness_and_pred = get_fitness_function(
        img_norm, label, model, mean, std, n_pixels
    )
    
    # Track optimization progress
    generation = [0]
    best_fitness = [float('inf')]
    best_candidate = [None]
    
    def wrapped_fitness(c):
        """Fitness wrapper to track best candidate."""
        fit = fitness_and_pred(c)[0]
        if fit < best_fitness[0]:
            best_fitness[0] = fit
            best_candidate[0] = c.copy()
        return fit
    
    def callback(xk, convergence):
        """Called after each generation. Returns True to stop early."""
        generation[0] += 1
        
        # Stop if we've successfully fooled the model
        if best_candidate[0] is not None:
            _, pred, _, _ = fitness_and_pred(best_candidate[0])
            if pred != label:
                return True  # Attack succeeded, no need to continue
        
        return False
    
    # Run differential evolution optimizer
    result = differential_evolution(
        wrapped_fitness,
        bounds,
        strategy="best1bin",
        maxiter=maxiter,
        popsize=popsize,
        mutation=(0.5, 1),
        recombination=1,
        callback=callback,
        atol=-1,
        tol=0.01,
        polish=False,
    )
    
    # Evaluate final adversarial image
    final_fit, final_pred, final_conf, adv_img = fitness_and_pred(result.x)
    
    # Parse pixel coordinates and RGB values from solution
    pixels = []
    for i in range(n_pixels):
        idx = i * 5
        x = int(round(result.x[idx]))
        y = int(round(result.x[idx + 1]))
        r = result.x[idx + 2]
        g = result.x[idx + 3]
        b = result.x[idx + 4]
        pixels.append((x, y, r, g, b))
    
    return pixels, result.nit, final_fit, final_pred, final_conf

def attack_image_n_pixels(model, img_norm, true_label, mean, std, 
                          n_pixels=1, popsize=80, maxiter=60):
    """Run n-pixel attack and return whether it succeeded."""
    
    # Run the differential evolution attack
    pixels, generations, final_fitness, adv_pred, adv_conf = differential_evolution_attack_n_pixels(
        img_norm, true_label, model, mean, std, 
        n_pixels=n_pixels, popsize=popsize, maxiter=maxiter
    )
    
    # Check if attack fooled the model (prediction changed)
    success = (adv_pred != true_label)
    
    return success, adv_pred, adv_conf, generations, final_fitness, pixels

In [22]:
# Shared helper functions
@torch.no_grad()
def get_baseline_prediction(model, img_norm, true_label):
    """Get the model's prediction and confidence for the true class."""
    
    # Get predicted class and its confidence
    pred, conf = predict(model, img_norm)
    
    # Get probability distribution over all classes
    probs = torch.softmax(model(img_norm.unsqueeze(0)), dim=1)[0]
    
    # Extract probability for the true class
    true_class_prob = probs[true_label].item()
    
    return pred, conf, true_class_prob

def normalize_image(img, mean, std):
    """Apply CIFAR-10 normalization to image."""
    return (img - mean.to(img.device)) / std.to(img.device)

def print_attack_start(image_num, true_label, pred, conf, true_class_prob):
    """Print header info before attacking an image."""
    true_name = CIFAR10_CLASSES[true_label]
    pred_name = CIFAR10_CLASSES[pred]
    print(f"\nAttacking image {image_num} | true = {true_name} | pred = {pred_name} ({conf:.3f})")
    print(f"Initial prob of TRUE class ({true_name}): {true_class_prob:.4f}")

def print_attack_result(success, true_label, adv_pred, adv_conf, pixels=None):
    """Print a short summary of the attack, with optional pixel details."""
    true_name = CIFAR10_CLASSES[true_label]
    adv_name = CIFAR10_CLASSES[adv_pred]

    if success:
        print(f"Attack succeeded: {true_name} → {adv_name} (confidence {adv_conf:.3f})")
    else:
        print(f"Attack failed: still {true_name} (confidence {adv_conf:.3f})")

def print_evaluation_summary(tried, success_count, total_gens_success):
    """Print final statistics after evaluation."""
    rate = (success_count / tried * 100) if tried > 0 else 0.0
    avg_gens = total_gens_success / success_count if success_count > 0 else 0.0
    
    print(f"\n{'='*60}")
    print(f"Finished {tried} correctly classified images.")
    print(f"Success rate: {rate:.2f}%")
    print(f"Average generations (successful attacks): {avg_gens:.2f}")
    print(f"{'='*60}\n")
    
    return rate

# Per-class evaluation helpers
def initialize_class_stats():
    """Initialize statistics tracking for all CIFAR-10 classes."""
    class_stats = {}
    for class_idx, class_name in enumerate(CIFAR10_CLASSES):
        class_stats[class_idx] = {
            'name': class_name,
            'success': 0,
            'total': 0,
            'total_gens': 0
        }
    return class_stats

def print_per_class_header(n_pixels, images_per_class):
    """Print header for per-class evaluation."""
    print(f"\n{'='*70}")
    print(f"Per-Class {n_pixels}-Pixel Attack Evaluation")
    print(f"Testing {images_per_class} correctly classified images per class")
    print(f"{'='*70}\n")

def print_attack_progress(true_name, current_count, images_per_class, conf):
    """Print progress indicator for current attack."""
    print(f"[{true_name} {current_count}/{images_per_class}] conf: {conf:.3f}", end=" ")

def update_class_stats(class_stats, true_label, success, gens):
    """Update statistics after an attack attempt."""
    if success:
        class_stats[true_label]['success'] += 1
        class_stats[true_label]['total_gens'] += gens

def print_per_class_summary(class_stats, n_pixels):
    """Print detailed per-class statistics table."""
    print(f"\n{'='*70}")
    print(f"Per-Class Attack Success Rate ({n_pixels}-pixel)")
    print(f"{'='*70}")
    print(f"{'Class':<15} {'Success':<12} {'Rate':<10} {'Avg Iters'}")
    print(f"{'-'*70}")
    
    total_success = 0
    total_tested = 0
    
    for class_idx in range(10):
        stats = class_stats[class_idx]
        name = stats['name'].capitalize()
        success = stats['success']
        total = stats['total']
        total_success += success
        total_tested += total
        
        if total > 0:
            rate = (success / total) * 100
            avg_gens = stats['total_gens'] / success if success > 0 else 0
            print(f"{name:<15} {success}/{total:<10} {rate:>5.1f}%     {avg_gens:>5.1f}")
        else:
            print(f"{name:<15} 0/0          -")
    
    print(f"{'-'*70}")
    overall_rate = (total_success / total_tested * 100) if total_tested > 0 else 0
    print(f"{'OVERALL':<15} {total_success}/{total_tested:<10} {overall_rate:>5.1f}%")
    print(f"{'='*70}\n")

# Main evaluation functions
@torch.no_grad()
def evaluate_attack_n_pixels(
    model,
    dataloader,
    mean,
    std,
    n_pixels=1,
    num_images=100,
    popsize=80,
    maxiter=60,
    label_filter=None,
):
    """
    Run n-pixel attack on test images with detailed per-image reporting.
    
    Args:
        model: Target model to attack
        dataloader: DataLoader yielding (image, label) pairs
        mean, std: Normalization statistics
        n_pixels: Number of pixels to perturb
        num_images: Maximum images to test
        popsize: DE optimizer population size
        maxiter: DE optimizer max generations
        label_filter: If set, only attack images with this true label
    
    Returns:
        Attack success rate as percentage
    """
    success_count = 0
    tried = 0
    total_gens_success = 0
    
    print(f"\n{'='*60}")
    print(f"Running {n_pixels}-PIXEL ATTACK")
    print(f"{'='*60}\n")
    
    for images, labels in dataloader:
        img = images[0].to(device)
        true_label = labels[0].item()
        
        # Skip if filtering by label
        if label_filter is not None and true_label != label_filter:
            continue
        
        # Normalize and get baseline prediction
        img_norm = normalize_image(img, mean, std)
        pred, conf, true_class_prob = get_baseline_prediction(model, img_norm, true_label)
        
        # Only attack correctly classified images
        if pred != true_label:
            continue
        
        tried += 1
        print_attack_start(tried, true_label, pred, conf, true_class_prob)
        
        # Execute attack
        success, adv_pred, adv_conf, gens, final_fit, pixels = attack_image_n_pixels(
            model, img_norm, true_label, mean, std, 
            n_pixels=n_pixels, popsize=popsize, maxiter=maxiter
        )
        
        print_attack_result(success, true_label, adv_pred, adv_conf, pixels)
        
        if success:
            success_count += 1
            total_gens_success += gens
        
        if tried >= num_images:
            break
    
    return print_evaluation_summary(tried, success_count, total_gens_success)

@torch.no_grad()
def evaluate_attack_per_class(
    model,
    dataloader,
    mean,
    std,
    n_pixels=1,
    images_per_class=10,
    popsize=80,
    maxiter=60,
):
    """
    Evaluate n-pixel attack success rate across all CIFAR-10 classes.
    
    Runs the attack on a fixed number of correctly classified images per class
    and reports per-class and overall success statistics.
    
    Args:
        model: Target model to attack
        dataloader: DataLoader yielding (image, label) pairs
        mean, std: Normalization statistics
        n_pixels: Number of pixels to perturb
        images_per_class: Number of images to test per class
        popsize: DE optimizer population size
        maxiter: DE optimizer max generations
    """
    class_stats = initialize_class_stats()
    print_per_class_header(n_pixels, images_per_class)
    
    # Track which classes have collected enough samples
    classes_complete = {i: False for i in range(10)}
    
    for images, labels in dataloader:
        # Stop when we've tested enough images from all classes
        if all(classes_complete.values()):
            break
        
        img = images[0].to(device)
        true_label = labels[0].item()
        
        # Skip if we've already tested enough from this class
        if class_stats[true_label]['total'] >= images_per_class:
            classes_complete[true_label] = True
            continue
        
        # Normalize and get prediction
        img_norm = normalize_image(img, mean, std)
        pred, conf = predict(model, img_norm)
        
        # Only attack images the model classifies correctly
        if pred != true_label:
            continue
        
        true_name = CIFAR10_CLASSES[true_label]
        class_stats[true_label]['total'] += 1
        current_count = class_stats[true_label]['total']
        
        print_attack_progress(true_name, current_count, images_per_class, conf)
        
        # Execute attack
        success, adv_pred, adv_conf, gens, final_fit, pixels = attack_image_n_pixels(
            model, img_norm, true_label, mean, std, 
            n_pixels=n_pixels, popsize=popsize, maxiter=maxiter
        )
        
        print_attack_result(success, true_label, adv_pred, adv_conf)
        update_class_stats(class_stats, true_label, success, gens)
    
    print_per_class_summary(class_stats, n_pixels)
    return class_stats

In [20]:
@torch.no_grad()
def evaluate_model(model, loader, device, mean, std):
    """Evaluate model accuracy on test set"""
    model.eval()
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # Normalize images
        images = (images - mean.to(device)) / std.to(device)
        
        logits = model(images)
        preds = logits.argmax(dim=1)
        
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total * 100
    print(f"Test accuracy: {acc:.2f}% ({correct}/{total})")
    return acc

# Evaluate model
evaluate_model(model, testloader, device, mean, std)

Test accuracy: 92.59% (9259/10000)


92.58999999999999

In [21]:
# Run 1-pixel attack
evaluate_attack_n_pixels(model, testloader, mean, std, n_pixels=1, num_images=100, popsize=80, maxiter=60)


Running 1-PIXEL ATTACK


Attacking image 1 | true = horse | pred = horse (0.985)
Initial prob of TRUE class (horse): 0.9848
Attack failed: still horse (confidence 0.549)

Attacking image 2 | true = dog | pred = dog (0.981)
Initial prob of TRUE class (dog): 0.9811


KeyboardInterrupt: 

In [ ]:
# Run 2-pixel attack
evaluate_attack_n_pixels(model, testloader, mean, std, n_pixels=2, num_images=10, popsize=40, maxiter=40)

In [ ]:
# Run 3-pixel attack
evaluate_attack_n_pixels(model, testloader, mean, std, n_pixels=3, num_images=10, popsize=40, maxiter=40)

In [ ]:
# Test 1-pixel attack: 5 images per class
stats_1pixel = evaluate_attack_per_class(
    model, testloader, mean, std,
    n_pixels=1,
    images_per_class=20,
    popsize=80,
    maxiter=40
)


Per-Class 1-Pixel Attack Evaluation
Testing 20 correctly classified images per class

[ship 1/20] conf: 0.974 Attack succeeded: ship → truck (confidence 0.542)
[horse 1/20] conf: 0.988 Attack failed: still horse (confidence 0.978)
[automobile 1/20] conf: 0.985 Attack failed: still automobile (confidence 0.975)
[horse 2/20] conf: 0.986 Attack succeeded: horse → dog (confidence 0.790)
[bird 1/20] conf: 0.993 Attack succeeded: bird → dog (confidence 0.596)
[airplane 1/20] conf: 0.987 Attack failed: still airplane (confidence 0.979)
[frog 1/20] conf: 0.980 Attack failed: still frog (confidence 0.962)
[dog 1/20] conf: 0.984 Attack succeeded: dog → cat (confidence 0.589)
[deer 1/20] conf: 0.988 Attack failed: still deer (confidence 0.987)
[dog 2/20] conf: 0.984 

In [ ]:
# Test 2-pixel attack: 10 images per class
stats_2pixel = evaluate_attack_per_class(
    model, testloader, mean, std,
    n_pixels=2,
    images_per_class=10,
    popsize=80,
    maxiter=80
)

In [ ]:
# Test 3-pixel attack: 10 images per class
stats_3pixel = evaluate_attack_per_class(
    model, testloader, mean, std,
    n_pixels=3,
    images_per_class=10,
    popsize=100,
    maxiter=100
)